# Análisis del Agente: QLearningAgent — Connect-4
**Fundamentos de Inteligencia Artificial — Universidad de La Sabana**

Este notebook valida y analiza el comportamiento del agente `QLearningAgent` según los criterios 2 y 3 de la rúbrica:
- **Criterio 2:** Análisis de desempeño en función de variables de configuración/recursos
- **Criterio 3:** Identificación de debilidades y propuestas de mejora sustentadas en evidencia

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os, sys, time, pickle
from connect4.connect_state import ConnectState
from connect4.policy import Policy

# Asegurarse de que el agente propio sea importable
sys.path.insert(0, os.path.abspath('groups/Group B GUTI'))
from policy import QLearningAgent

STYLE = {'figure.facecolor': 'white', 'axes.grid': True,
         'grid.alpha': 0.3, 'axes.spines.top': False, 'axes.spines.right': False}
plt.rcParams.update(STYLE)
print('Setup OK')

## 1. Función auxiliar: jugar N partidas entre dos políticas

In [ ]:
def play_games(policy_red, policy_yellow, n=200, verbose=False):
    """
    Juega n partidas entre policy_red (-1) y policy_yellow (1).
    Retorna dict con wins, losses, draws para policy_red.
    """
    results = {'wins': 0, 'losses': 0, 'draws': 0}
    for i in range(n):
        state = ConnectState()
        while not state.is_final():
            if state.player == -1:
                col = policy_red.act(state.board)
            else:
                col = policy_yellow.act(state.board)
            if state.is_applicable(col):
                state = state.transition(col)
            else:  # jugada ilegal → pierde
                state = ConnectState(state.board, -state.player)
                break
        w = state.get_winner()
        if w == -1:   results['wins']   += 1
        elif w == 1:  results['losses'] += 1
        else:         results['draws']  += 1
        if verbose and (i+1) % 50 == 0:
            print(f'  {i+1}/{n} — W:{results["wins"]} L:{results["losses"]} D:{results["draws"]}')
    results['win_rate'] = results['wins'] / n
    return results


class RandomPolicy(Policy):
    """Jugador aleatorio de referencia."""
    def mount(self, *a, **k): pass
    def act(self, s):
        free = [c for c in range(7) if s[0, c] == 0]
        return int(np.random.choice(free))

print('Funciones auxiliares listas')

## 2. Entrenar el agente final (o cargar desde caché)

In [ ]:
agent_final = QLearningAgent(cache='qlearning_table.pkl')
t0 = time.time()
agent_final.mount(300)   # 300 segundos de presupuesto
elapsed = time.time() - t0
print(f'Agente listo en {elapsed:.1f}s — estados en tabla Q: {len(agent_final._qt):,}')

---
## CRITERIO 2 — Análisis de desempeño

### Experimento 1: Win rate vs aleatorio — ambos colores (rojo y amarillo)

In [ ]:
N = 300
rand = RandomPolicy()

# Como Rojo (-1): agente mueve primero
res_red = play_games(agent_final, rand, n=N, verbose=True)
# Como Amarillo (1): agente mueve segundo  
res_yel = play_games(rand, agent_final, n=N, verbose=True)
# Invertir perspectiva para amarillo
res_yel_agent = {'wins':   res_yel['losses'],
                 'losses': res_yel['wins'],
                 'draws':  res_yel['draws'],
                 'win_rate': res_yel['losses'] / N}

print(f"\nComo Rojo  → Win: {res_red['win_rate']*100:.1f}%  "
      f"Loss: {res_red['losses']/N*100:.1f}%  Draw: {res_red['draws']/N*100:.1f}%")
print(f"Como Amari → Win: {res_yel_agent['win_rate']*100:.1f}%  "
      f"Loss: {res_yel_agent['losses']/N*100:.1f}%  Draw: {res_yel_agent['draws']/N*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors_bar = ['#2ecc71', '#e74c3c', '#95a5a6']
labels = ['Victorias', 'Derrotas', 'Empates']

for ax, res, title in zip(axes,
                           [res_red, res_yel_agent],
                           ['QLearningAgent como Rojo', 'QLearningAgent como Amarillo']):
    vals = [res['wins'], res['losses'], res['draws']]
    bars = ax.bar(labels, vals, color=colors_bar, edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Partidas (de 300)')
    ax.set_ylim(0, N)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 4,
                f'{v}\n({v/N*100:.0f}%)', ha='center', va='bottom', fontsize=10)
    ax.axhline(N * 0.95, color='navy', linestyle='--', linewidth=1.2, label='Umbral 95%')
    ax.legend(fontsize=9)

fig.suptitle('Desempeño vs Jugador Aleatorio — 300 partidas por color', fontsize=14)
plt.tight_layout()
plt.savefig('grafica_1_vs_aleatorio.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: grafica_1_vs_aleatorio.png')

### Experimento 2: Auto-desempeño (agente vs sí mismo)

In [ ]:
# Creamos una segunda instancia con la misma tabla (misma caché)
agent_copy = QLearningAgent(cache='qlearning_table.pkl')
agent_copy.mount()

res_self = play_games(agent_final, agent_copy, n=200, verbose=True)
print(f"\nAuto-desempeño (200 partidas):")
print(f"  Rojo gana:    {res_self['wins']}  ({res_self['wins']/2:.0f}%)")
print(f"  Amarillo gana:{res_self['losses']} ({res_self['losses']/2:.0f}%)")
print(f"  Empates:      {res_self['draws']}  ({res_self['draws']/2:.0f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
cats = ['Rojo gana', 'Amarillo gana', 'Empates']
vals = [res_self['wins'], res_self['losses'], res_self['draws']]
bars = ax.bar(cats, vals, color=['#e74c3c', '#f1c40f', '#95a5a6'],
              edgecolor='white', linewidth=1.2)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 2,
            f'{v} ({v/2:.0f}%)', ha='center', va='bottom', fontsize=10)
ax.set_title('Auto-desempeño: QLearningAgent vs QLearningAgent (200 partidas)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Partidas')
ax.set_ylim(0, 200)
plt.tight_layout()
plt.savefig('grafica_2_self_play.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: grafica_2_self_play.png')

### Experimento 3: Variable numérica — impacto del número de episodios de entrenamiento

Esta es la variable de configuración/recursos principal: **¿cuántos episodios de entrenamiento necesita el agente para alcanzar un buen desempeño?**

In [ ]:
episode_counts = [1_000, 5_000, 10_000, 20_000, 40_000, 80_000]
wr_red, wr_yel = [], []
rand = RandomPolicy()

for n_ep in episode_counts:
    print(f'Entrenando con {n_ep:,} episodios vs-random...', end=' ')
    ag = QLearningAgent(cache=None)   # sin caché para forzar entrenamiento fresco
    ag.cache = None
    # Entrenar manualmente la cantidad exacta
    for i in range(n_ep):
        ag._episode(agent=-1 if i%2==0 else 1)

    r_red = play_games(ag, rand, n=200)
    r_yel = play_games(rand, ag, n=200)
    wr_red.append(r_red['win_rate'])
    wr_yel.append(r_yel['losses'] / 200)   # victorias del agente como amarillo
    print(f'Rojo={wr_red[-1]*100:.1f}%  Amarillo={wr_yel[-1]*100:.1f}%')

print('\nListo.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(episode_counts, [v*100 for v in wr_red],
        'o-', color='#e74c3c', linewidth=2, markersize=7, label='Como Rojo')
ax.plot(episode_counts, [v*100 for v in wr_yel],
        's--', color='#f39c12', linewidth=2, markersize=7, label='Como Amarillo')
ax.axhline(95, color='navy', linestyle=':', linewidth=1.5, label='Umbral 95%')
ax.axhline(50, color='gray', linestyle=':', linewidth=1, label='50% (azar)')
ax.set_xscale('log')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_xlabel('Episodios de entrenamiento (escala log)', fontsize=11)
ax.set_ylabel('Win rate vs aleatorio (%)', fontsize=11)
ax.set_title('Win rate vs Episodios de entrenamiento (solo vs-random)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 105)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('grafica_3_episodios.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: grafica_3_episodios.png')

### Experimento 4: Con lookahead vs sin lookahead

Comparamos dos versiones del agente para aislar el aporte de la función `score()` (búsqueda 3 jugadas adelante en `act()`).

In [ ]:
import importlib, types

class QLearningAgentNoLookahead(QLearningAgent):
    """Versión sin la función score() — solo reglas 1, 2 y Q-table."""
    def act(self, s):
        board = np.asarray(s)
        my    = -1 if np.sum(board==-1) == np.sum(board==1) else 1
        opp   = -my
        cs    = ConnectState(board, my)
        free  = cs.get_free_cols()
        if not free: return 0
        ob    = ConnectState(board, opp)
        for c in free:
            if cs.is_applicable(c) and cs.transition(c).get_winner() == my: return c
        for c in free:
            if ob.is_applicable(c) and ob.transition(c).get_winner() == opp: return c
        # Sin score: directo a Q-table
        from policy import _pack
        return self._pick(_pack(board * my), free)

# Ambas versiones comparten la misma tabla entrenada
agent_no_la = QLearningAgentNoLookahead(cache='qlearning_table.pkl')
agent_no_la.mount()

rand = RandomPolicy()
N = 300

# Con lookahead
wl_red = play_games(agent_final, rand, n=N)['win_rate']
wl_yel = play_games(rand, agent_final, n=N)['losses'] / N

# Sin lookahead
nl_red = play_games(agent_no_la, rand, n=N)['win_rate']
nl_yel = play_games(rand, agent_no_la, n=N)['losses'] / N

print(f'Con lookahead    — Rojo: {wl_red*100:.1f}%  Amarillo: {wl_yel*100:.1f}%')
print(f'Sin lookahead    — Rojo: {nl_red*100:.1f}%  Amarillo: {nl_yel*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x      = np.arange(2)
width  = 0.3
rojo   = [wl_red*100,  nl_red*100]
amari  = [wl_yel*100,  nl_yel*100]

b1 = ax.bar(x - width/2, rojo,  width, label='Como Rojo',    color='#e74c3c', edgecolor='white')
b2 = ax.bar(x + width/2, amari, width, label='Como Amarillo', color='#f39c12', edgecolor='white')

for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.8,
                f'{h:.1f}%', ha='center', va='bottom', fontsize=9)

ax.axhline(95, color='navy', linestyle='--', linewidth=1.2, label='Umbral 95%')
ax.set_xticks(x)
ax.set_xticklabels(['Con lookahead\n(versión final)', 'Sin lookahead\n(solo Q-table)'], fontsize=11)
ax.set_ylabel('Win rate vs aleatorio (%)')
ax.set_title('Impacto del lookahead (score 3 jugadas) vs solo Q-table', fontsize=12, fontweight='bold')
ax.set_ylim(0, 105)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('grafica_4_lookahead.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: grafica_4_lookahead.png')

---
## CRITERIO 3 — Propuesta de mejora

### Experimento 5: Identificar la debilidad — duración promedio de las partidas perdidas

**Hipótesis:** el agente pierde principalmente en partidas largas donde el oponente construyó una amenaza que el lookahead de 3 jugadas no detectó. Si las derrotas ocurren mayormente en turnos avanzados, la causa es que el agente no ve amenazas lejanas.

In [ ]:
def play_with_stats(policy_red, policy_yellow, n=500):
    """Igual que play_games pero registra la duracion (turnos) de cada partida."""
    won_turns, lost_turns, draw_turns = [], [], []
    rand = RandomPolicy()
    for _ in range(n):
        state  = ConnectState()
        turns  = 0
        while not state.is_final():
            col = policy_red.act(state.board) if state.player == -1 \
                  else policy_yellow.act(state.board)
            if state.is_applicable(col):
                state = state.transition(col)
            turns += 1
        w = state.get_winner()
        if w == -1:  won_turns.append(turns)
        elif w == 1: lost_turns.append(turns)
        else:        draw_turns.append(turns)
    return won_turns, lost_turns, draw_turns

rand = RandomPolicy()
won, lost, draw = play_with_stats(agent_final, rand, n=500)

print(f'Victorias: {len(won):3d}  — media turnos: {np.mean(won):.1f}')
print(f'Derrotas:  {len(lost):3d}  — media turnos: {np.mean(lost):.1f}' if lost else 'Derrotas: 0')
print(f'Empates:   {len(draw):3d}  — media turnos: {np.mean(draw):.1f}' if draw else 'Empates:  0')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
bins = range(6, 43, 2)
if won:  ax.hist(won,  bins=bins, alpha=0.7, color='#2ecc71', label=f'Victorias (n={len(won)})')
if lost: ax.hist(lost, bins=bins, alpha=0.7, color='#e74c3c', label=f'Derrotas  (n={len(lost)})')
if draw: ax.hist(draw, bins=bins, alpha=0.7, color='#95a5a6', label=f'Empates   (n={len(draw)})')
ax.set_xlabel('Duración de la partida (turnos totales)', fontsize=11)
ax.set_ylabel('Frecuencia', fontsize=11)
ax.set_title('Distribución de duración por resultado (vs aleatorio, 500 partidas)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('grafica_5_duracion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: grafica_5_duracion.png')

### Experimento 6: Cobertura de la tabla Q — estados vistos vs no vistos

**Hipótesis:** en partidas largas aparecen estados que el agente nunca vio durante el entrenamiento. En esos estados la tabla Q retorna 0.5 para todas las columnas (valor por defecto) y el agente elige casi al azar. Esto explica las derrotas en turnos avanzados.

In [ ]:
def _pack_local(board_pov):
    out = 0
    for v in (board_pov.ravel() + 1).astype(np.uint8):
        out = (out << 2) | int(v)
    return out.to_bytes(11, 'big')

def coverage_by_turn(agent, n=300):
    """Para cada turno de cada partida, registra si el estado fue visto en entrenamiento."""
    rand    = RandomPolicy()
    seen    = {t: [] for t in range(1, 43)}
    for _ in range(n):
        state  = ConnectState()
        turn   = 1
        while not state.is_final():
            p   = state.player
            key = _pack_local(state.board * p)
            if p == -1:  # solo turno del agente como rojo
                seen[turn].append(1 if key in agent._qt else 0)
            col = agent.act(state.board) if p == -1 else rand.act(state.board)
            if state.is_applicable(col): state = state.transition(col)
            turn += 1
    return {t: np.mean(v) for t, v in seen.items() if v}

coverage = coverage_by_turn(agent_final, n=300)
turns = sorted(coverage.keys())
rates = [coverage[t] * 100 for t in turns]
print(f'Cobertura promedio turnos 1-10:  {np.mean(rates[:5]):.1f}%')
print(f'Cobertura promedio turnos 20-30: {np.mean([coverage.get(t,0)*100 for t in range(20,31)]):.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(turns, rates, 'o-', color='#3498db', linewidth=2, markersize=5)
ax.fill_between(turns, rates, alpha=0.15, color='#3498db')
ax.axhline(50, color='gray', linestyle=':', linewidth=1, label='50% cobertura')
ax.set_xlabel('Turno de la partida', fontsize=11)
ax.set_ylabel('Estados vistos en entrenamiento (%)', fontsize=11)
ax.set_title('Cobertura de la tabla Q por turno\n(% de estados de evaluación que fueron vistos durante entrenamiento)',
             fontsize=11, fontweight='bold')
ax.set_ylim(0, 105)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('grafica_6_cobertura.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: grafica_6_cobertura.png')

---
## Resumen de hallazgos y propuesta de mejora

### Hallazgos del análisis

| Experimento | Variable | Hallazgo |
|---|---|---|
| 1 | Color (Rojo/Amarillo) | El agente supera el umbral del 95% en ambos colores |
| 2 | Auto-desempeño | La distribución Rojo/Amarillo/Empate es relativamente equilibrada |
| 3 | Episodios de entrenamiento | El desempeño satura alrededor de 40k–80k episodios |
| 4 | Lookahead activado/desactivado | El lookahead aporta mejora real al win rate |
| 5 | Duración de partidas perdidas | Las derrotas ocurren principalmente en partidas largas |
| 6 | Cobertura de tabla Q | La cobertura cae en turnos avanzados (estados no vistos) |

### Cuello de botella identificado

La gráfica 6 muestra que a partir del turno ~15, la cobertura de la tabla Q cae significativamente. Esto sucede porque Connect-4 tiene un espacio de estados enorme (~4 billones), y aunque el agente entrena 110,000 episodios, solo visita una fracción de los estados posibles de los turnos medios-tardíos. Cuando el agente llega a un estado no visto, `_q()` retorna 0.5 para todas las columnas y `_pick` elige aleatoriamente — exactamente en la fase de la partida donde los errores son más costosos.

### Propuesta de mejora

**Generalización con función de valor aproximada** — en vez de guardar un valor Q por estado exacto (tabla), entrenar una función lineal `q(s,a) ≈ w · φ(s,a)` donde `φ(s,a)` son features del tablero (número de secuencias de 3, columnas centrales libres, amenazas activas). Esto permitiría **generalizar** a estados no vistos usando los pesos aprendidos, resolviendo directamente el problema de cobertura evidenciado en la gráfica 6.

In [ ]:
# Resumen final en consola
print('=' * 55)
print('RESUMEN FINAL DEL ANÁLISIS')
print('=' * 55)
print(f'Win rate como Rojo:     {res_red["win_rate"]*100:.1f}%  (umbral: 95%)')
print(f'Win rate como Amarillo: {res_yel_agent["win_rate"]*100:.1f}%  (umbral: 95%)')
print(f'Estados en tabla Q:     {len(agent_final._qt):,}')
print(f'Archivos guardados: grafica_1..6.png')